# 🚀 Stack 2.9 - Kaggle Training

Free GPU training on Kaggle using Qwen2.5-Coder-7B.

⏱️ **Runtime:** 2-4 hours  |  💾 **VRAM:** ~14GB (bfloat16, no bitsandbytes)

**Setup:**
1. Settings → Accelerator → GPU **T4**
2. Run all cells in order
3. Download merged model from Output tab when done

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone repository
import os, shutil, subprocess

os.chdir('/kaggle/working')
REPO_DIR = '/kaggle/working/stack-2.9'
OUTPUT_DIR = os.path.join(REPO_DIR, 'training_output')

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', 'https://github.com/my-ai-stack/stack-2.9.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
print('✅ Repo ready:', REPO_DIR)

In [ ]:
# Save to Kaggle output (download before session ends!)
# Kaggle sessions expire after 9 hours - download outputs immediately!

# Create a symbolic link to make paths easier
OUTPUT_DIR = os.path.join(REPO_DIR, 'training_output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"✅ Output directory: {OUTPUT_DIR}")
print("⚠️ IMPORTANT: Download outputs from 'Output' tab before session expires!")


In [ ]:
# Install PyTorch (force CUDA 11.8 build for sm_60 Pascal GPU compatibility)
# Kaggle sometimes assigns P100 (sm_60) which requires CUDA 11.x builds of PyTorch
!pip uninstall -y torch torchvision torchaudio
!pip install torch==2.2.0+cu118 torchvision==0.17.0+cu118 torchaudio==2.2.0+cu118 --index-url https://download.pytorch.org/whl/cu118
print('✅ PyTorch ready')

In [ ]:
# Install other dependencies (NO bitsandbytes — bfloat16 only)
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.34.0 datasets==3.0.0 pyyaml tqdm scipy numpy
print('✅ Dependencies ready')

In [ ]:
# Fix NumPy 2.0 compatibility (downgrade to <2.0)
!pip install -q "numpy<2" --force-reinstall
print('✅ NumPy downgraded to <2.0')


In [ ]:
# Prepare training data (auto-detect or synthetic fallback)
import os, json

REPO_TRAIN_DATA = os.path.join(REPO_DIR, 'training-data/final/train.jsonl')
MINI_DATA_DIR = os.path.join(REPO_DIR, 'data_mini')
MINI_DATA_FILE = os.path.join(MINI_DATA_DIR, 'train_mini.jsonl')
SYNTHETIC_FILE = os.path.join(REPO_DIR, 'data/synthetic.jsonl')

print('🔍 Data check')

if os.path.exists(REPO_TRAIN_DATA):
    os.makedirs(MINI_DATA_DIR, exist_ok=True)
    if not os.path.exists(MINI_DATA_FILE):
        print('   Building mini dataset (1K samples) from full data...')
        !python scripts/create_mini_dataset.py --size 1000 --output {MINI_DATA_FILE} --source {REPO_TRAIN_DATA}
    DATA_FILE = MINI_DATA_FILE
    print('   Using mini dataset')
elif os.path.exists(MINI_DATA_FILE):
    DATA_FILE = MINI_DATA_FILE
    print('   Using existing mini dataset')
else:
    print('   Creating synthetic data (last resort)')
    examples = [
        {'instruction': 'Write a Python function to reverse a string', 'output': 'def reverse_string(s):\n    return s[::-1]'},
        {'instruction': 'Write a function to check if a number is prime', 'output': 'def is_prime(n):\n    if n <= 1:\n        return False\n    for i in range(2, int(n**0.5) + 1):\n        if n % i == 0:\n            return False\n        return True'},
        {'instruction': 'Write a binary search function', 'output': 'def binary_search(arr, target):\n    left, right = 0, len(arr) - 1\n    while left <= right:\n        mid = (left + right) // 2\n        if arr[mid] == target:\n            return mid\n        elif arr[mid] < target:\n            left = mid + 1\n        else:\n            right = mid - 1\n        return -1'},
    ]
    samples = examples * 333
    os.makedirs(os.path.dirname(SYNTHETIC_FILE), exist_ok=True)
    with open(SYNTHETIC_FILE, 'w') as f:
        for s in samples:
            f.write(json.dumps(s) + '\n')
    DATA_FILE = SYNTHETIC_FILE
    print(f'   Synthetic dataset: {len(samples)} examples')

print(f'\n✅ Data: {DATA_FILE}')
!ls -lh {DATA_FILE}

In [ ]:
# Generate training configuration
# Uses bfloat16 only (NO bitsandbytes — avoids CUDA 13 dependency issues)
import yaml

os.makedirs(OUTPUT_DIR, exist_ok=True)

config = {
    'model': {'name': 'Qwen/Qwen2.5-Coder-1.5B', 'trust_remote_code': True},
    'data': {'input_path': DATA_FILE, 'max_length': 2048, 'train_split': 0.999},
    'lora': {'r': 8, 'lora_alpha': 16, 'dropout': 0.05, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], 'bias': 'none', 'task_type': 'CAUSAL_LM'},
    'training': {'num_epochs': 1, 'batch_size': 1, 'gradient_accumulation': 4, 'learning_rate': 2e-4, 'warmup_steps': 50, 'weight_decay': 0.01, 'max_grad_norm': 1.0, 'logging_steps': 10, 'save_steps': 100, 'save_total_limit': 2, 'fp16': True, 'bf16': False, 'gradient_checkpointing': True},
    'output': {'lora_dir': os.path.join(OUTPUT_DIR, 'lora'), 'logging_dir': os.path.join(OUTPUT_DIR, 'logs')},
    'quantization': {'enabled': False},
    'hardware': {'device': 'cuda', 'num_gpus': 1, 'use_4bit': False, 'use_8bit': False}
}

config_path = os.path.join(OUTPUT_DIR, 'train_config.yaml')
with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f'✅ Config: {config_path}')
print(f"   Model: {config['model']['name']}")
print(f"   Data: {config['data']['input_path']}")
print(f"   bf16={config['training']['bf16']}, fp16={config['training']['fp16']}")

In [ ]:
# Train (using standalone train_simple_nobnb.py - bfloat16, no quantization)
print('='*60)
print('STARTING TRAINING (bfloat16, no quantization)')
print('='*60)

!cd {REPO_DIR} && python train_simple_nobnb.py --config {config_path}

print('\n✅ Training step finished')

In [ ]:
# Merge LoRA adapter into final model
lora_dir = os.path.join(OUTPUT_DIR, 'lora')
merged_dir = os.path.join(OUTPUT_DIR, 'merged')

print('='*60)
print('MERGING LORA ADAPTER')
print('='*60)

!cd {REPO_DIR} && python merge_simple.py \
    --base-model {config['model']['name']} \
    --adapter-path {lora_dir} \
    --output-path {merged_dir} \
    --use-safetensors

print('\n✅ Merge complete!')
print(f'Merged model: {merged_dir}')
!ls -lh {merged_dir}

print("\n⚠️ DOWNLOAD THE MODEL NOW: Go to Output tab and download 'merged' folder!")


In [ ]:
# Push merged model to GitHub LFS (optional - for permanent storage)
# This saves the model to your GitHub repo so you can download anytime

# Configure Git LFS
!git lfs install 2>/dev/null || echo 'Git LFS already installed'

# Clone the repo if not already there
import subprocess
repo_url = 'https://github.com/my-ai-stack/stack-2.9.git'
local_repo = '/kaggle/working/stack-2.9-repo'

if not os.path.exists(local_repo):
    subprocess.run(['git', 'clone', repo_url, local_repo], check=True)

# Copy merged model to repo
import shutil
target_dir = os.path.join(local_repo, 'models/stack-2.9-finetuned')
os.makedirs(target_dir, exist_ok=True)

if os.path.exists(merged_dir):
    # Copy files
    for f in os.listdir(merged_dir):
        src = os.path.join(merged_dir, f)
        dst = os.path.join(target_dir, f)
        if os.path.isdir(src):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        else:
            shutil.copy2(src, dst)
    
    print(f'✅ Copied model to {target_dir}')
    
    # Push to GitHub
    os.chdir(local_repo)
    subprocess.run(['git', 'add', 'models/stack-2.9-finetuned/'], check=True)
    subprocess.run(['git', 'config', 'user.email', 'kaggle@kaggle.com'], check=True)
    subprocess.run(['git', 'config', 'user.name', 'Kaggle Auto-Push'], check=True)
    subprocess.run(['git', 'commit', '-m', 'feat: add fine-tuned model from Kaggle'], check=True)
    
    # Push (you may need a GitHub token for private repos)
    result = subprocess.run(['git', 'push', 'origin', 'main'], capture_output=True, text=True)
    if result.returncode == 0:
        print('✅ Model pushed to GitHub!')
    else:
        print(f'⚠️ Push failed: {result.stderr}')
        print('   You can still download from Kaggle Output tab.')
else:
    print('⚠️ Merged model not found. Train first!')


## 📥 Download Model

1. Open **Output** tab on the right
2. Find `training_output/merged/`
3. Select all files and **Download**

⚠️ **Do this before Kaggle session ends!**